#  Nexalyze — Synthetic Data Generation

This notebook generates **25 interconnected datasets** for the Enterprise Retail AI Intelligence Platform.

| Dataset | Records |
|---|---|
| Customers | 10,000 |
| Products | 5,000 |
| Transactions | 300,000 |
| Browsing History | 900,000 |
| Wishlist | 100,000 |
| Shopping Cart | 100,000 |
| Search History | 200,000 |
| Customer Sessions | 150,000 |
| Inventory | 25,000 |
| + 16 more... | — |

In [ ]:
# Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'faker', 'pandas', 'numpy', '-q'])

In [ ]:
import sys
from pathlib import Path

# Add notebooks directory to path so we can import generate_data
notebook_dir = Path().resolve()
sys.path.insert(0, str(notebook_dir))

print(f'Notebook directory: {notebook_dir}')
print(f'Data will be saved to: {notebook_dir.parent / "data" / "raw"}')

In [ ]:
# Run the full data generation pipeline
import generate_data
generate_data.main()

In [ ]:
# Validation: Check all files were created
import pandas as pd
from pathlib import Path

raw_dir = Path().resolve().parent / 'data' / 'raw'
expected = [
    'stores','warehouses','suppliers','employees','holiday_calendar','weather',
    'customers','products','transactions','browsing_history','customer_sessions',
    'wishlist','shopping_cart','search_history','inventory','inventory_movements',
    'pricing_history','competitor_pricing','promotions','marketing_campaigns',
    'orders','shipments','support_tickets','returns','customer_reviews'
]

print(f"{'Dataset':<25} {'Rows':>10} {'Cols':>6} {'Size':>10}")
print('-' * 55)
all_ok = True
for name in expected:
    fpath = raw_dir / f'{name}.csv'
    if fpath.exists():
        df = pd.read_csv(fpath, nrows=0)
        n_rows = sum(1 for _ in open(fpath)) - 1
        size_kb = fpath.stat().st_size / 1024
        print(f'{name:<25} {n_rows:>10,} {len(df.columns):>6} {size_kb:>8.0f} KB')
    else:
        print(f'{name:<25}   MISSING ')
        all_ok = False

print()
print(' All datasets generated!' if all_ok else ' Some datasets are missing — re-run generation cell.')

In [ ]:
# ── Referential Integrity Check ───────────────────────────────────────────
import pandas as pd
from pathlib import Path

raw = Path().resolve().parent / 'data' / 'raw'
read = lambda n: pd.read_csv(raw / f'{n}.csv')

customers_ids  = set(read('customers')['customer_id'])
product_ids    = set(read('products')['product_id'])
store_ids      = set(read('stores')['store_id'])

txn = read('transactions')
assert txn['customer_id'].isin(customers_ids).all(), 'BAD: transaction customer_id'
assert txn['product_id'].isin(product_ids).all(),   'BAD: transaction product_id'

bh = read('browsing_history')
assert bh['customer_id'].isin(customers_ids).all(), 'BAD: browsing customer_id'
assert bh['product_id'].isin(product_ids).all(),   'BAD: browsing product_id'

print(' Referential integrity checks passed!')

In [ ]:
# ── Quick Preview ──────────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

raw = Path().resolve().parent / 'data' / 'raw'
for name in ['customers', 'products', 'transactions']:
    print(f'\n=== {name.upper()} ===')
    display(pd.read_csv(raw / f'{name}.csv').head(3))